# Import Libraries

In [ ]:
import pandas as pd
import numpy as np

import tensorflow as tf
import spacy
import transformers
from sklearn.preprocessing import LabelEncoder

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

# Data Loading & Exploration

In [ ]:
data=pd.read_csv('News_Category_Dataset_v3.csv')

In [ ]:
data

In [ ]:
df=pd.DataFrame(data)
# data.head()

In [ ]:
df_sampled=df.sample(n=30000,random_state=42)
df_sampled.shape

# Data Cleaning & Preprocessing

In [ ]:
df

In [ ]:
df=df_sampled.drop(['Unnamed: 0','link','date','authors'],axis='columns')

In [ ]:
df['short_description'][df['short_description'].isna()].count
# df.loc[df['headline'].isna(), 'headline']


In [ ]:
# df['headline'] = df['headline'].fillna('')
df['short_description'] = df['short_description'].fillna('')
df['text']=df['headline'].astype(str)+" "+df['short_description'].astype(str)
df.head()

In [ ]:
pip install langdetect

In [ ]:
import re
import spacy
from langdetect import detect

nlp = spacy.load("en_core_web_sm")

def clean_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r'http\S+|www\S+|@\S+', '', text)  # Remove URLs/emails
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = " ".join(text.split())  # Remove extra spaces

    # Lemmatization
    doc = nlp(text)
    text = " ".join([token.lemma_ for token in doc if not token.is_stop])


    return text

df['cleaned_text'] = df['text'].apply(clean_text)

# NLP Processing

In [ ]:
# def remove_punct(text):
#     doc=nlp(text)
#     return ' '.join([token.text for token in doc if not token.is_punct ])

# df['cleaned_txt']=df['cleaned_txt'].apply(remove_punct)

In [ ]:
df['cleaned_text'][df['cleaned_text'].isna()].count

In [ ]:
df.head()

In [ ]:
labels_le=LabelEncoder()
df['category']=labels_le.fit_transform(df['category'])

In [ ]:
df.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(dtype=np.float32)
X = vectorizer.fit_transform(df['cleaned_text'])
y = df['category']

# Feature Engineering using TF-IDF

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = Sequential([
    Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(len(labels_le.classes_), activation='softmax')
])
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# from sklearn.model_selection import train_test_split
# from sklearn.feature_extraction.text import TfidfVectorizer

# # Initialize vectorizer
# vectorizer = TfidfVectorizer()
# X = vectorizer.fit_transform(df['cleaned_text'])  # Keep sparse
# y = df['category']

# # Split into train and test
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Fit model (TensorFlow supports sparse input)
# model.fit(X_train, y_train, epochs=10, batch_size=20, validation_data=(X_test, y_test))


# Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# Assuming df is already loaded and preprocessed

# Initialize vectorizer
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['cleaned_text'])  # Keep sparse
y = df['category']

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define a simple neural network model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(len(set(y)), activation='softmax')  # Output layer with number of classes
])

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Fit model with early stopping
model.fit(X_train, y_train, epochs=50, batch_size=64, validation_data=(X_test, y_test), callbacks=[early_stopping])


In [ ]:
import pickle

with open("vectorizer7.pkl", "wb") as v_file:
    pickle.dump(vectorizer, v_file)

with open("label_encoder7.pkl", "wb") as l_file:
    pickle.dump(labels_le, l_file)


# Neural Network Model

In [ ]:
import numpy as np

# Example text
sample_text = ['''
Prime Minister Narendra Modi
''']

# Transform using TF-IDF
sample_tfidf = vectorizer.transform(sample_text).toarray()

# Predict category probabilities
predicted_probabilities = model.predict(sample_tfidf)  # Get probability distribution

# Get indices of the top 4 categories (sorted from highest to lowest)
top_4_indices = np.argsort(predicted_probabilities[0])[-4:][::-1]  # Sort & take top 4

# Convert indices to category names
top_4_categories = labels_le.inverse_transform(top_4_indices)

print("Top 4 Predicted Categories:", top_4_categories)


# Model Training

In [ ]:
model.save('model50thoushand.h5')

In [ ]:
import joblib
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(labels_le, 'label_encoder.pkl')

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import numpy as np

model = load_model("model20thoushand.h5")
with open("tfidf_vectorizer.pkl", "rb") as f:
    joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

with open("label_encoder.pkl", "rb") as f:
    joblib.dump(labels_le, 'label_encoder.pkl')

raw_text = ['''

Right-wing commentator and writer Ann Coulter questioned the arrest and the following discussion over the possibility of deporting Mahmoud Khalil, the Columbia University student who was arrested for pro-Palestinian protests on campuses last year. Khalil was arrested by the ICE (Immigration and Customs Enforcement) over the weekend and even President Donald Trump made a social media post mentioning Khalil's name -- blowing the issue out of proportion as his administration is trying to make an example out of Khalil. Secretary of State Marco Rubio said visas or green cards of such Hamas supporters will be revoked so that they can be deported.
Khalil's lawyer Amy Greer said Khalil is a green card holder and he was arrested even after she mentioned the legality of his citizenship to the administration.
“There’s almost no one I don’t want to deport, but unless they’ve committed a crime, isn’t this a violation of the First Amendment?” Coulter said on the social platform X in response to a report from the New York Post. The First Amendment is the right to free speech.


''']

text_tfidf = vectorizer.transform(raw_text)
prediction = model.predict(text_tfidf)

top_4_indices = np.argsort(prediction[0])[-4:][::-1]
top_4_categories = labels_le.inverse_transform(top_4_indices)
top_4_probabilities = prediction[0][top_4_indices]

print("Predicted Categories (Top 4):")
for category, prob in zip(top_4_categories, top_4_probabilities):
    print(f"{category}: {prob*100:.2f}%")


# Model Evaluation

In [ ]:
import pandas as pd

# ✅ Sample News Data (Only Short Descriptions, 3 Lines Each)
news_data = [
    ["A leading tech company introduced a groundbreaking AI tool. It aims to improve automation in business operations. Experts say it could revolutionize multiple industries."],

    ["The stock market reached an all-time high this week. Strong earnings reports and investor confidence fueled the surge. Analysts predict further gains in the coming months."],

    ["A new vaccine has shown 95% effectiveness in early trials. Scientists say it could help control future outbreaks. Health organizations are fast-tracking its approval."],

    ["World leaders gathered to discuss climate change policies. The main focus is reducing global carbon emissions. Activists are calling for more urgent action."],

    ["A top football player has signed a record-breaking contract. The deal is worth over $300 million over five years. Fans are excited about his future performances."],

    ["NASA successfully landed a spacecraft on Mars. The mission aims to study the planet’s atmosphere and soil. Scientists hope to find signs of ancient life."],

    ["Governments worldwide are considering cryptocurrency regulations. Officials worry about financial crimes and market instability. Investors fear stricter rules may limit innovation."],

    ["Economists warn about rising inflation rates globally. The cost of living has increased in many countries. Central banks are discussing possible interest rate hikes."],

    ["Scientists have developed a potential cancer breakthrough. A new treatment shows promising results in early trials. Experts believe it could lead to higher survival rates."],

    ["Electric vehicle sales have surged in the past year. Governments are offering incentives for switching to clean energy. Automakers are investing heavily in battery technology."],

    ["A major cyberattack targeted a multinational bank. Customers faced disruptions in online transactions for hours. Authorities are investigating the source of the attack."],

    ["A new blockbuster film has broken box office records. Fans rushed to theaters to watch the highly anticipated sequel. Critics praise its stunning visuals and storyline."],

    ["Millions of citizens are voting in the national elections. The outcome could shape the country’s future policies. Political analysts predict a tight race between candidates."],

    ["Tech companies are laying off employees amid economic uncertainty. The industry is facing slow growth and declining revenues. Experts suggest companies are adjusting to market changes."],

    ["Gold prices have reached a five-year high. Economic instability has driven investors toward safer assets. Analysts expect the trend to continue in the coming months."],

    ["Governments debate banning certain social media platforms. Officials cite concerns over misinformation and data privacy. Critics argue that such bans violate free speech rights."],

    ["Companies are replacing human workers with AI systems. Automation is increasing in industries like retail and finance. Experts warn of potential job losses in the future."],

    ["Severe wildfires have forced thousands to evacuate. Emergency responders are struggling to contain the flames. Climate experts link the fires to rising global temperatures."],

    ["A new diabetes treatment may eliminate the need for insulin. Early research suggests it could revolutionize diabetes care. Doctors are optimistic but call for more clinical trials."],

    ["Global supply chain disruptions continue to affect industries. Delays in shipping and manufacturing are causing shortages. Companies are struggling to meet consumer demand."],

    ["Housing prices are rising rapidly in major cities. Many first-time buyers find it hard to afford homes. Experts suggest policy changes to address the crisis."],

    ["A major tech company is facing an antitrust lawsuit. Regulators accuse it of engaging in anti-competitive practices. The case could reshape the industry’s future regulations."],

    ["Streaming platforms are fiercely competing for subscribers. New content releases and exclusive deals dominate the market. Consumers are benefiting from diverse entertainment choices."],

    ["A new smartphone with innovative features has launched. The device includes AI-powered camera enhancements. Tech enthusiasts are eager to test its performance."],

    ["Unemployment rates have dropped as job markets recover. Businesses are hiring more workers after pandemic-related struggles. Economists say the rebound is a positive sign."],

    ["Retailers are testing drone deliveries in major cities. The new technology aims to speed up e-commerce shipping. Regulators are reviewing safety guidelines before expansion."],

    ["Sustainable fashion is gaining popularity worldwide. More brands are using eco-friendly materials in production. Consumers are increasingly supporting ethical clothing choices."],

    ["Athletes are preparing for the upcoming Olympic Games. Countries are finalizing their teams for various sports. The event is expected to attract millions of viewers globally."],

    ["Scientists have developed a new eco-friendly battery. It could help reduce reliance on lithium-ion technology. The breakthrough may lead to more sustainable energy storage."],

    ["Space agencies are planning a mission to explore Jupiter. The spacecraft will study the planet’s atmosphere and moons. Scientists hope to uncover secrets of the solar system."]
]

# Convert Data to Pandas DataFrame
df = pd.DataFrame(news_data, columns=["News_Description"])

# Save to Excel File
df.to_excel("news_data.xlsx", index=False)

print("excel file 'news_data.xlsx' has been created successfully!")


In [ ]:
data=pd.read_excel('news_data.xlsx')

In [ ]:
df=pd.DataFrame(data)
df.head()

# Model Saving & Prediction

In [ ]:
df.shape